In [1]:
# ============================================================
# CELL 1: Installs & Imports
# ============================================================
!pip install -q kagglehub yfinance doubleml

import kagglehub, os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_1samp, ttest_ind
from doubleml import DoubleMLData, DoubleMLPLR
from sklearn.ensemble import RandomForestRegressor
warnings.filterwarnings('ignore')
np.random.seed(42)

In [2]:
# ============================================================
# CELL 2: Load & Clean Data
# ============================================================
path = kagglehub.dataset_download("heresjohnnyv/congress-investments")
df = pd.read_csv(path + "/SenatorCleaned.csv")

stocks = (df
    .drop(columns=["Unnamed: 0"], errors="ignore")
    .assign(**{"Transaction.Date": lambda x: pd.to_datetime(x["Transaction.Date"], errors="coerce")})
    .query("`Asset.Type` == 'Stock' and Ticker != '--'")
    .query("Type in ['Purchase', 'Sale (Full)', 'Sale (Partial)']")
    .copy()
)
stocks["purchase"] = (stocks["Type"] == "Purchase").astype(int)
stocks["Name"] = stocks["Name"].str.strip()
print(stocks.shape)

(12259, 10)


In [3]:
# ============================================================
# CELL 3: Senator Metadata (Party, Committee, Demographics)
# Source: govinfo.gov congressional records (113th-117th Congress)
#         Wikipedia biographical articles
# ============================================================
political_party_map = {
    'A. Mitchell McConnell Jr.': 1, 'Angus S King Jr.': 2,
    'Barbara A Mikulski': 0, 'Benjamin L Cardin': 0,
    'Chris Van Hollen': 0, 'Christopher A Coons': 0,
    'Claire McCaskill': 0, 'Cory A Booker': 0,
    'Daniel R Coats': 1, 'Daniel S Sullivan': 1,
    'David A Perdue Jr': 1, 'David B Vitter': 1,
    'Dean Heller': 1, 'Gary C Peters': 0,
    'Jacklyn S Rosen': 0, 'James M Inhofe': 1,
    'Jeffry L Flake': 1, 'Jerry Moran': 1,
    'John Cornyn': 1, 'John D Rockefeller IV': 0,
    'John F Reed': 0, 'John Hoeven': 1,
    'John N Kennedy': 1, 'John W Hickenlooper': 0,
    'Jon Kyl': 1, 'Joseph Manchin III': 0,
    'Kelly Loeffler': 1, 'Ladda Tammy Duckworth': 0,
    'Maria Cantwell': 0, 'Mark Begich': 0,
    'Mark R Warner': 0, 'Michael B Enzi': 1,
    'Michael F Bennet': 0, 'Pat Roberts': 1,
    'Patrick J Toomey': 1, 'Patty Murray': 0,
    'Rafael E Cruz': 1, 'Rand Paul': 1,
    'Robert P Casey Jr.': 0, 'Robert P Corker Jr.': 1,
    'Roy Blunt': 1, 'Sheldon Whitehouse': 0,
    'Shelley M Capito': 1, 'Susan M Collins': 1,
    'Thad Cochran': 1, 'Thomas H Tuberville': 1,
    'Thomas R Carper': 0, 'Thomas R Tillis': 1,
    'Thomas Udall': 0, 'Timothy M Kaine': 0,
    'Tina Smith': 0, 'Tom Coburn': 1,
    'William Cassidy': 1, 'William F Hagerty IV': 1
}

committee_map = {
    'A. Mitchell McConnell Jr.': {'finance':0,'banking':0,'armed_services':0,'energy':0,'intelligence':0,'appropriations':1},
    'Angus S King Jr.':          {'finance':0,'banking':0,'armed_services':1,'energy':1,'intelligence':1,'appropriations':0},
    'Barbara A Mikulski':        {'finance':0,'banking':0,'armed_services':0,'energy':0,'intelligence':0,'appropriations':1},
    'Benjamin L Cardin':         {'finance':0,'banking':0,'armed_services':0,'energy':0,'intelligence':0,'appropriations':0},
    'Chris Van Hollen':          {'finance':0,'banking':1,'armed_services':0,'energy':0,'intelligence':0,'appropriations':1},
    'Christopher A Coons':       {'finance':0,'banking':0,'armed_services':0,'energy':0,'intelligence':0,'appropriations':1},
    'Claire McCaskill':          {'finance':0,'banking':0,'armed_services':1,'energy':0,'intelligence':0,'appropriations':0},
    'Cory A Booker':             {'finance':0,'banking':1,'armed_services':0,'energy':0,'intelligence':0,'appropriations':0},
    'Daniel R Coats':            {'finance':0,'banking':0,'armed_services':0,'energy':0,'intelligence':1,'appropriations':0},
    'Daniel S Sullivan':         {'finance':0,'banking':0,'armed_services':1,'energy':1,'intelligence':0,'appropriations':0},
    'David A Perdue Jr':         {'finance':0,'banking':1,'armed_services':1,'energy':0,'intelligence':0,'appropriations':0},
    'David B Vitter':            {'finance':0,'banking':1,'armed_services':0,'energy':0,'intelligence':0,'appropriations':0},
    'Dean Heller':               {'finance':1,'banking':1,'armed_services':0,'energy':0,'intelligence':0,'appropriations':0},
    'Gary C Peters':             {'finance':0,'banking':0,'armed_services':1,'energy':0,'intelligence':1,'appropriations':0},
    'Jacklyn S Rosen':           {'finance':0,'banking':0,'armed_services':1,'energy':0,'intelligence':1,'appropriations':0},
    'James M Inhofe':            {'finance':0,'banking':0,'armed_services':1,'energy':0,'intelligence':0,'appropriations':0},
    'Jeffry L Flake':            {'finance':0,'banking':0,'armed_services':0,'energy':1,'intelligence':0,'appropriations':0},
    'Jerry Moran':               {'finance':0,'banking':1,'armed_services':0,'energy':0,'intelligence':1,'appropriations':1},
    'John Cornyn':               {'finance':1,'banking':0,'armed_services':0,'energy':0,'intelligence':1,'appropriations':0},
    'John D Rockefeller IV':     {'finance':1,'banking':0,'armed_services':1,'energy':1,'intelligence':1,'appropriations':0},
    'John F Reed':               {'finance':0,'banking':1,'armed_services':1,'energy':0,'intelligence':0,'appropriations':1},
    'John Hoeven':               {'finance':0,'banking':0,'armed_services':0,'energy':1,'intelligence':0,'appropriations':1},
    'John N Kennedy':            {'finance':1,'banking':0,'armed_services':0,'energy':0,'intelligence':0,'appropriations':1},
    'John W Hickenlooper':       {'finance':0,'banking':0,'armed_services':0,'energy':1,'intelligence':0,'appropriations':0},
    'Jon Kyl':                   {'finance':1,'banking':0,'armed_services':0,'energy':0,'intelligence':1,'appropriations':0},
    'Joseph Manchin III':        {'finance':0,'banking':0,'armed_services':1,'energy':1,'intelligence':1,'appropriations':0},
    'Kelly Loeffler':            {'finance':0,'banking':0,'armed_services':1,'energy':0,'intelligence':0,'appropriations':0},
    'Ladda Tammy Duckworth':     {'finance':0,'banking':0,'armed_services':1,'energy':0,'intelligence':0,'appropriations':0},
    'Maria Cantwell':            {'finance':1,'banking':0,'armed_services':0,'energy':1,'intelligence':1,'appropriations':0},
    'Mark Begich':               {'finance':0,'banking':1,'armed_services':1,'energy':0,'intelligence':0,'appropriations':1},
    'Mark R Warner':             {'finance':1,'banking':1,'armed_services':0,'energy':0,'intelligence':1,'appropriations':0},
    'Michael B Enzi':            {'finance':1,'banking':0,'armed_services':0,'energy':0,'intelligence':0,'appropriations':0},
    'Michael F Bennet':          {'finance':1,'banking':0,'armed_services':0,'energy':0,'intelligence':1,'appropriations':0},
    'Pat Roberts':               {'finance':1,'banking':0,'armed_services':0,'energy':0,'intelligence':0,'appropriations':0},
    'Patrick J Toomey':          {'finance':1,'banking':1,'armed_services':0,'energy':0,'intelligence':0,'appropriations':0},
    'Patty Murray':              {'finance':0,'banking':0,'armed_services':0,'energy':0,'intelligence':0,'appropriations':1},
    'Rafael E Cruz':             {'finance':0,'banking':0,'armed_services':0,'energy':0,'intelligence':0,'appropriations':0},
    'Rand Paul':                 {'finance':0,'banking':0,'armed_services':0,'energy':0,'intelligence':0,'appropriations':0},
    'Robert P Casey Jr.':        {'finance':1,'banking':0,'armed_services':0,'energy':0,'intelligence':0,'appropriations':0},
    'Robert P Corker Jr.':       {'finance':0,'banking':1,'armed_services':0,'energy':1,'intelligence':0,'appropriations':0},
    'Roy Blunt':                 {'finance':0,'banking':0,'armed_services':0,'energy':0,'intelligence':1,'appropriations':1},
    'Sheldon Whitehouse':        {'finance':1,'banking':0,'armed_services':0,'energy':0,'intelligence':0,'appropriations':0},
    'Shelley M Capito':          {'finance':0,'banking':0,'armed_services':0,'energy':1,'intelligence':0,'appropriations':1},
    'Susan M Collins':           {'finance':0,'banking':0,'armed_services':0,'energy':0,'intelligence':1,'appropriations':1},
    'Thad Cochran':              {'finance':0,'banking':0,'armed_services':0,'energy':0,'intelligence':0,'appropriations':1},
    'Thomas H Tuberville':       {'finance':0,'banking':0,'armed_services':1,'energy':0,'intelligence':0,'appropriations':0},
    'Thomas R Carper':           {'finance':1,'banking':0,'armed_services':0,'energy':0,'intelligence':0,'appropriations':0},
    'Thomas R Tillis':           {'finance':1,'banking':1,'armed_services':0,'energy':0,'intelligence':0,'appropriations':0},
    'Thomas Udall':              {'finance':0,'banking':0,'armed_services':0,'energy':0,'intelligence':0,'appropriations':1},
    'Timothy M Kaine':           {'finance':0,'banking':0,'armed_services':1,'energy':0,'intelligence':0,'appropriations':0},
    'Tina Smith':                {'finance':1,'banking':0,'armed_services':0,'energy':1,'intelligence':0,'appropriations':0},
    'Tom Coburn':                {'finance':0,'banking':0,'armed_services':0,'energy':0,'intelligence':1,'appropriations':0},
    'William Cassidy':           {'finance':1,'banking':0,'armed_services':0,'energy':1,'intelligence':0,'appropriations':0},
    'William F Hagerty IV':      {'finance':0,'banking':0,'armed_services':0,'energy':0,'intelligence':0,'appropriations':0},
}

senator_demographics = {
    'A. Mitchell McConnell Jr.': {'birth_year':1942,'female':0},
    'Angus S King Jr.':          {'birth_year':1944,'female':0},
    'Barbara A Mikulski':        {'birth_year':1936,'female':1},
    'Benjamin L Cardin':         {'birth_year':1943,'female':0},
    'Chris Van Hollen':          {'birth_year':1959,'female':0},
    'Christopher A Coons':       {'birth_year':1963,'female':0},
    'Claire McCaskill':          {'birth_year':1953,'female':1},
    'Cory A Booker':             {'birth_year':1969,'female':0},
    'Daniel R Coats':            {'birth_year':1943,'female':0},
    'Daniel S Sullivan':         {'birth_year':1964,'female':0},
    'David A Perdue Jr':         {'birth_year':1949,'female':0},
    'David B Vitter':            {'birth_year':1961,'female':0},
    'Dean Heller':               {'birth_year':1960,'female':0},
    'Gary C Peters':             {'birth_year':1958,'female':0},
    'Jacklyn S Rosen':           {'birth_year':1957,'female':1},
    'James M Inhofe':            {'birth_year':1934,'female':0},
    'Jeffry L Flake':            {'birth_year':1962,'female':0},
    'Jerry Moran':               {'birth_year':1954,'female':0},
    'John Cornyn':               {'birth_year':1952,'female':0},
    'John D Rockefeller IV':     {'birth_year':1937,'female':0},
    'John F Reed':               {'birth_year':1949,'female':0},
    'John Hoeven':               {'birth_year':1957,'female':0},
    'John N Kennedy':            {'birth_year':1951,'female':0},
    'John W Hickenlooper':       {'birth_year':1952,'female':0},
    'Jon Kyl':                   {'birth_year':1942,'female':0},
    'Joseph Manchin III':        {'birth_year':1947,'female':0},
    'Kelly Loeffler':            {'birth_year':1970,'female':1},
    'Ladda Tammy Duckworth':     {'birth_year':1968,'female':1},
    'Maria Cantwell':            {'birth_year':1958,'female':1},
    'Mark Begich':               {'birth_year':1962,'female':0},
    'Mark R Warner':             {'birth_year':1954,'female':0},
    'Michael B Enzi':            {'birth_year':1944,'female':0},
    'Michael F Bennet':          {'birth_year':1964,'female':0},
    'Pat Roberts':               {'birth_year':1936,'female':0},
    'Patrick J Toomey':          {'birth_year':1961,'female':0},
    'Patty Murray':              {'birth_year':1950,'female':1},
    'Rafael E Cruz':             {'birth_year':1970,'female':0},
    'Rand Paul':                 {'birth_year':1963,'female':0},
    'Robert P Casey Jr.':        {'birth_year':1960,'female':0},
    'Robert P Corker Jr.':       {'birth_year':1952,'female':0},
    'Roy Blunt':                 {'birth_year':1950,'female':0},
    'Sheldon Whitehouse':        {'birth_year':1955,'female':0},
    'Shelley M Capito':          {'birth_year':1953,'female':1},
    'Susan M Collins':           {'birth_year':1952,'female':1},
    'Thad Cochran':              {'birth_year':1937,'female':0},
    'Thomas H Tuberville':       {'birth_year':1964,'female':0},
    'Thomas R Carper':           {'birth_year':1947,'female':0},
    'Thomas R Tillis':           {'birth_year':1960,'female':0},
    'Thomas Udall':              {'birth_year':1948,'female':0},
    'Timothy M Kaine':           {'birth_year':1958,'female':0},
    'Tina Smith':                {'birth_year':1958,'female':1},
    'Tom Coburn':                {'birth_year':1948,'female':0},
    'William Cassidy':           {'birth_year':1957,'female':0},
    'William F Hagerty IV':      {'birth_year':1959,'female':0},
}

# Build lookup DataFrames
party_df = pd.DataFrame.from_dict(
    {k: {'party': v} for k, v in political_party_map.items()}, orient='index'
).reset_index().rename(columns={'index': 'Name'})

committee_df = pd.DataFrame.from_dict(committee_map, orient='index').reset_index()
committee_df.columns = ['Name', 'finance', 'banking', 'armed_services', 'energy', 'intelligence', 'appropriations']

demo_df = pd.DataFrame.from_dict(senator_demographics, orient='index').reset_index()
demo_df.columns = ['Name', 'birth_year', 'female']

# Merge all into one senator_meta table
senator_meta = party_df.merge(committee_df, on='Name').merge(demo_df, on='Name')
senator_meta['party_label'] = senator_meta['party'].map({0:'Democrat',1:'Republican',2:'Independent'})
print(senator_meta.shape)
senator_meta.head()

(54, 11)


,Name,party,finance,banking,armed_services,energy,intelligence,appropriations,birth_year,female,party_label
0,A. Mitchell McConnell Jr.,1,0,0,0,0,0,1,1942,0,Republican
1,Angus S King Jr.,2,0,0,1,1,1,0,1944,0,Independent
2,Barbara A Mikulski,0,0,0,0,0,0,1,1936,1,Democrat
3,Benjamin L Cardin,0,0,0,0,0,0,0,1943,0,Democrat
4,Chris Van Hollen,0,0,1,0,0,0,1,1959,0,Democrat


In [4]:
# ============================================================
# CELL 4: Filter to R/D only and merge metadata
# ============================================================
stocks["Name"] = stocks["Name"].str.strip()
stocks_rd = stocks[stocks["Name"].isin(senator_meta[senator_meta["party"].isin([0,1])]["Name"])].copy()
stocks_rd = stocks_rd.merge(senator_meta, on='Name', how='left')
print(stocks_rd.shape)

(12228, 20)


In [5]:
# ============================================================
# CELL 5: Download Price Data (top 250 tickers + SPY)
# ============================================================
import yfinance as yf

trades = stocks_rd.copy()
trades["Transaction.Date"] = pd.to_datetime(trades["Transaction.Date"], errors="coerce")
trades["Ticker"] = trades["Ticker"].str.strip().str.upper()
trades = trades[trades["Transaction.Date"].notna() & trades["Ticker"].notna() & (trades["Ticker"] != "--")].copy()

start_date = trades["Transaction.Date"].min() - pd.Timedelta(days=10)
end_date   = trades["Transaction.Date"].max() + pd.Timedelta(days=45)

top_tickers = trades["Ticker"].value_counts().head(250).index.tolist()
print(f"Downloading {len(top_tickers)} tickers...")

price_data = {}
for ticker in top_tickers:
    try:
        hist = yf.download(ticker, start=start_date.strftime("%Y-%m-%d"),
                           end=end_date.strftime("%Y-%m-%d"),
                           auto_adjust=True, progress=False, threads=False)
        if not hist.empty:
            temp = hist[["Close"]].reset_index()
            temp.columns = ["Date", "Close"]
            temp["Ticker"] = ticker
            price_data[ticker] = temp
    except:
        pass

prices = pd.concat(price_data.values(), ignore_index=True)
prices["Date"] = pd.to_datetime(prices["Date"]).astype("datetime64[s]")
prices = prices.sort_values(["Ticker", "Date"]).reset_index(drop=True)

# SPY benchmark
spy = yf.download("SPY", start="2012-01-01", end="2021-12-31",
                  auto_adjust=True, progress=False, threads=False)
spy = spy[["Close"]].reset_index()
spy.columns = ["Date", "Close"]
spy["Date"] = pd.to_datetime(spy["Date"]).astype("datetime64[s]")
spy = spy.sort_values("Date").reset_index(drop=True)

print(f"Good tickers: {len(price_data)}")

$DISCA: possibly delisted; no timezone found

1 Failed download:
['DISCA']: possibly delisted; no timezone found
$FEYE: possibly delisted; no timezone found

1 Failed download:
['FEYE']: possibly delisted; no timezone found
$CBS: possibly delisted; no timezone found

1 Failed download:
['CBS']: possibly delisted; no timezone found
$WPX: possibly delisted; no timezone found

1 Failed download:
['WPX']: possibly delisted; no timezone found
$FDC: possibly delisted; no timezone found

1 Failed download:
['FDC']: possibly delisted; no timezone found
$FB: possibly delisted; no price data found  (1d 2011-11-19 -> 2021-09-13) (Yahoo error = "Data doesn't exist for startDate = 1321678800, endDate = 1631505600")

1 Failed download:
['FB']: possibly delisted; no price data found  (1d 2011-11-19 -> 2021-09-13) (Yahoo error = "Data doesn't exist for startDate = 1321678800, endDate = 1631505600")
$AAN: possibly delisted; no timezone found

1 Failed download:
['AAN']: possibly delisted; no timezone f

Good tickers: 201


In [9]:
# ============================================================
# CELL 6: Build Returns (entry, 30d exit, SPY benchmark)
# ============================================================
trades_sub = trades[trades["Ticker"].isin(price_data.keys())].copy()
trades_sub["Transaction.Date"] = trades_sub["Transaction.Date"].astype("datetime64[s]")
trades_sub = trades_sub.sort_values(["Transaction.Date", "Ticker"]).reset_index(drop=True)

# Entry price
entry = pd.merge_asof(trades_sub,
                      prices.sort_values(["Date", "Ticker"]).reset_index(drop=True),
                      left_on="Transaction.Date", right_on="Date",
                      by="Ticker", direction="forward"
                     ).rename(columns={"Date": "entry_date", "Close": "entry_price"})
# 30-day exit price
entry["target_date_30"] = (entry["Transaction.Date"] + pd.Timedelta(days=30)).astype("datetime64[s]")
future_30 = pd.merge_asof(entry.sort_values(["target_date_30","Ticker"]).reset_index(drop=True),
                           prices.sort_values(["Date","Ticker"]).reset_index(drop=True),
                           left_on="target_date_30", right_on="Date",
                           by="Ticker", direction="forward"
                          ).rename(columns={"Date": "exit_date_30", "Close": "exit_price_30"})

# Stock return
future_30["return_30d"] = (future_30["exit_price_30"] - future_30["entry_price"]) / future_30["entry_price"]

# SPY entry
future_30 = future_30.sort_values("Transaction.Date").reset_index(drop=True)
future_30 = pd.merge_asof(future_30,
                           spy.rename(columns={"Date":"spy_entry_date","Close":"spy_entry_price"}),
                           left_on="Transaction.Date", right_on="spy_entry_date", direction="forward")

# SPY exit
future_30 = future_30.sort_values("target_date_30").reset_index(drop=True)
future_30 = pd.merge_asof(future_30,
                           spy.rename(columns={"Date":"spy_exit_date","Close":"spy_exit_price_30"}),
                           left_on="target_date_30", right_on="spy_exit_date", direction="forward")

# Excess and directional returns
future_30["spy_return_30d_exact"] = (future_30["spy_exit_price_30"] - future_30["spy_entry_price"]) / future_30["spy_entry_price"]
future_30["excess_return_30d"]    = future_30["return_30d"] - future_30["spy_return_30d_exact"]
future_30["directional_excess"]   = np.where(future_30["Type"] == "Purchase",
                                              future_30["excess_return_30d"],
                                             -future_30["excess_return_30d"])

# Drop unmatched rows and add DML controls
future_30 = future_30[future_30[["spy_entry_price","spy_exit_price_30","excess_return_30d"]].notna().all(axis=1)].copy()
future_30["year"] = future_30["Transaction.Date"].dt.year
future_30["senator_trade_volume"] = future_30.groupby("Name")["directional_excess"].transform("count")
future_30["age_at_trade"] = future_30["year"] - future_30["birth_year"]

print(future_30.shape)
future_30[["Name","party_label","directional_excess","excess_return_30d","year","finance","banking","female","age_at_trade"]].head()

(6690, 36)


,Name,party_label,directional_excess,excess_return_30d,year,finance,banking,female,age_at_trade
0,Thomas R Carper,Democrat,-0.077290,-0.077290,2012,1,0,0,65
1,Thomas R Carper,Democrat,0.038299,0.038299,2012,1,0,0,65
2,Jeffry L Flake,Republican,0.008619,-0.008619,2013,0,0,0,51
3,Thomas R Carper,Democrat,-0.093873,-0.093873,2013,1,0,0,66
4,Thomas R Carper,Democrat,-0.026754,-0.026754,2013,1,0,0,66


In [13]:
!pip install -q doubleml

import warnings
warnings.filterwarnings('ignore')

from doubleml import DoubleMLData, DoubleMLPLR
from sklearn.ensemble import RandomForestRegressor

dml_cols = ['directional_excess', 'party', 'year', 'senator_trade_volume', 'purchase',
            'finance', 'banking', 'armed_services', 'energy', 'intelligence', 
            'appropriations', 'Amount', 'female', 'age_at_trade']

dml_df = future_30[dml_cols].dropna().copy()
print(f"DML sample size: {len(dml_df)}")

dml_data = DoubleMLData(
    dml_df,
    y_col='directional_excess',
    d_cols='party',
    x_cols=['year', 'senator_trade_volume', 'purchase', 'finance', 'banking',
            'armed_services', 'energy', 'intelligence', 'appropriations',
            'Amount', 'female', 'age_at_trade']
)

ml_l = RandomForestRegressor(n_estimators=500, max_depth=7, random_state=42)
ml_m = RandomForestRegressor(n_estimators=500, max_depth=7, random_state=42)

dml_plr = DoubleMLPLR(dml_data, ml_l, ml_m, n_folds=5)
dml_plr.fit()

print(dml_plr.summary)
print(f"\nATE: {dml_plr.coef[0]:.4f}")
print(f"95% CI: [{dml_plr.confint().iloc[0,0]:.4f}, {dml_plr.confint().iloc[0,1]:.4f}]")

# Sensitivity analysis (same as lab)
dml_plr.sensitivity_analysis()
print(dml_plr.sensitivity_summary)

DML sample size: 6690
           coef   std err         t     P>|t|     2.5 %    97.5 %
party  0.001542  0.021631  0.071283  0.943172 -0.040853  0.043937

ATE: 0.0015
95% CI: [-0.0409, 0.0439]
================== Sensitivity Analysis ==================

------------------ Scenario          ------------------
Significance Level: level=0.95
Sensitivity parameters: cf_y=0.03; cf_d=0.03, rho=1.0

------------------ Bounds with CI    ------------------
       CI lower  theta lower     theta  theta upper  CI upper
party -0.101178    -0.064092  0.001542     0.067176   0.10384

------------------ Robustness Values ------------------
       H_0    RV (%)   RVa (%)
party  0.0  0.071391  0.000502


In [11]:
from scipy.stats import ttest_ind

for committee in ['finance', 'banking', 'armed_services', 'energy', 'intelligence', 'appropriations']:
    on = future_30[future_30[committee] == 1]['directional_excess'].dropna()
    off = future_30[future_30[committee] == 0]['directional_excess'].dropna()
    t, p = ttest_ind(on, off)
    print(f"{committee:15s}  on={on.mean():.4f}  off={off.mean():.4f}  p={p:.4f}")

finance          on=-0.0012  off=0.0025  p=0.1360
banking          on=0.0044  off=-0.0037  p=0.0001
armed_services   on=0.0035  off=0.0006  p=0.1583
energy           on=0.0026  off=0.0012  p=0.4797
intelligence     on=-0.0071  off=0.0020  p=0.0914
appropriations   on=-0.0018  off=0.0023  p=0.1378


In [14]:
# DML with Banking committee membership as treatment
dml_cols_banking = ['directional_excess', 'banking', 'year', 'senator_trade_volume', 'purchase',
                    'finance', 'armed_services', 'energy', 'intelligence', 
                    'appropriations', 'Amount', 'female', 'age_at_trade', 'party']

dml_df_banking = future_30[dml_cols_banking].dropna().copy()
print(f"DML sample size: {len(dml_df_banking)}")

dml_data_banking = DoubleMLData(
    dml_df_banking,
    y_col='directional_excess',
    d_cols='banking',
    x_cols=['year', 'senator_trade_volume', 'purchase', 'finance', 
            'armed_services', 'energy', 'intelligence', 'appropriations',
            'Amount', 'female', 'age_at_trade', 'party']
)

np.random.seed(42)
ml_l2 = RandomForestRegressor(n_estimators=500, max_depth=7, random_state=42)
ml_m2 = RandomForestRegressor(n_estimators=500, max_depth=7, random_state=42)

dml_banking = DoubleMLPLR(dml_data_banking, ml_l2, ml_m2, n_folds=5)
dml_banking.fit()

print(dml_banking.summary)
print(f"\nATE: {dml_banking.coef[0]:.4f}")
print(f"95% CI: [{dml_banking.confint().iloc[0,0]:.4f}, {dml_banking.confint().iloc[0,1]:.4f}]")

dml_banking.sensitivity_analysis()
print(dml_banking.sensitivity_summary)

DML sample size: 6690
             coef   std err         t     P>|t|     2.5 %    97.5 %
banking -0.015026  0.027267 -0.551059  0.581593 -0.068467  0.038416

ATE: -0.0150
95% CI: [-0.0685, 0.0384]
================== Sensitivity Analysis ==================

------------------ Scenario          ------------------
Significance Level: level=0.95
Sensitivity parameters: cf_y=0.03; cf_d=0.03, rho=1.0

------------------ Bounds with CI    ------------------
         CI lower  theta lower     theta  theta upper  CI upper
banking -0.139173    -0.089793 -0.015026     0.059742  0.104595

------------------ Robustness Values ------------------
         H_0    RV (%)   RVa (%)
banking  0.0  0.610433  0.000435
